<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">5. Lakehouse Federation</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# 5.3 Lecture: Migration to UC using a Federated Catalog

This lesson covers catalog federation as a bridge for migrating from legacy metastores like Hive Metastore and AWS Glue into Unity Catalog without disrupting existing workloads.

## Learning Objectives

By the end of this lesson, you will be able to:
- Describe how catalog federation serves as a bridge for migration to Unity Catalog
- Compare the pros and cons of federated catalog access
- Explain the steps for setting up foreign catalogs from AWS Glue
- Describe the in-place migration path from foreign to managed tables
- Identify testing and validation strategies for migration

## A. Context: Migration to UC Using Federation

Federation lets UC act as a read-through layer over an existing metastore so that users can query the legacy catalog through UC while individual tables are migrated in stages.

<div class="mermaid" id="diagram-5-3-migration-federation" style="font-size: 1em;">
flowchart LR
    Legacy["<b>Legacy Catalogs</b><br/>HMS, Glue"] -->|"Federation"| UC["<b>Unity Catalog</b><br/>Unified Governance"]
    UC --> Managed["<b>Managed Tables</b><br/>Full UC Features"]
    Legacy -.->|"Phased<br/>Migration"| Managed
    style UC fill:#FF3621,stroke:#CC2B1A,stroke-width:2px,color:#fff
    style Managed fill:#e8f5e9,stroke:#4caf50,stroke-width:2px
    style Legacy fill:#fff3e0,stroke:#ff9800,stroke-width:2px
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: false });
await new Promise(r => requestAnimationFrame(r));
try {
  await mermaid.run({ querySelector: "#diagram-5-3-migration-federation" });
} catch(e) {
  await new Promise(r => setTimeout(r, 1000));
  await mermaid.run({ querySelector: "#diagram-5-3-migration-federation" });
}
document.querySelectorAll('#diagram-5-3-migration-federation svg text, #diagram-5-3-migration-federation svg .nodeLabel, #diagram-5-3-migration-federation svg foreignObject div, #diagram-5-3-migration-federation svg span').forEach(el => { el.style.fontSize = '1em'; });
</script>

- Establish a **migration path** from legacy catalogs (HMS, AWS Glue) to Unity Catalog
- Maintain **operational continuity** during the migration process
- Leverage federation as a **bridge technology** for phased migration
- Gradually **consolidate governance** while minimizing disruption

## B. Pros and Cons of Federation as a Migration Path

| Pros | Cons |
|------|------|
| Access to UC governance features while maintaining existing workflows | Performance overhead when querying across federated catalogs |
| Zero-downtime migration with minimal disruption | Added complexity of managing multiple catalog systems |
| Cross-catalog queries enable analytics across both legacy and new assets | Many UC features only available for fully migrated tables |

## C. Setting up Foreign Catalogs

Foreign catalogs are configured by chaining together IAM roles, storage credentials, a connection object, and the foreign catalog itself.

### Foreign Catalog from AWS Glue to Unity Catalog

The four-step sequence below sets up a foreign catalog that mirrors a Glue database into UC.

1. **Create IAM Role(s)** in AWS for access to Glue and underlying S3 storage
2. **Create Storage and Service Credentials** in Databricks using the role(s)
3. **Create a Connection Object** in Databricks
4. **Create the Foreign Catalog** mirroring the Glue database

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Flexibility</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The approach varies by provider. The same pattern works for Hive Metastore, Snowflake, and other federated sources.</p>
        </div>
    </div>
</div>

## D. Migration to Managed Tables

Federation is intended as a temporary state: the long-term goal is to convert foreign tables into UC-managed tables to unlock the full UC feature set.

### Why Migrate from Foreign to Managed?

The benefits below are the main drivers for completing the migration rather than staying in the federated state.

| Benefit | Detail |
|---------|--------|
| **Full UC Feature Set** | Optimization features, quality monitoring, performance improvements |
| **Read/Write Support** | Convert read-only foreign tables to read/write managed tables |
| **Enhanced Governance** | Centralized metadata management with unified policies |
| **Simplified Administration** | No need to maintain synchronization with external catalogs |
| **Performance Gains** | Significant improvements over foreign tables |


### Migration Commands

Two `ALTER TABLE` statements move a table from the foreign state to external and then to managed.

<div class="code-block" data-language="sql">
-- Convert foreign table to external
ALTER TABLE <foreign_table> SET EXTERNAL;
<br/>
-- Convert external table to managed
ALTER TABLE <external_table> SET MANAGED;
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
    });
})();
</script>

### In-Place Migration Advantages

In-place migration is the preferred approach because it leaves table identifiers, ACLs, and downstream queries unchanged.

- **Namespace Preservation:** Same table identifiers in queries
- **Permission Continuity:** ACLs stay intact through migration
- **Minimal Code Changes:** No need to update downstream queries
- **Reduced Risk:** Ability to roll back if needed
- **Zero Downtime:** Production workloads continue during migration

## E. Testing and Validation

Before committing to a migration, dry runs and validation checks let you confirm compatibility without changing any table state.

<div style="font-size: 1em; border-left: 4px solid #607d8b; background: #eceff1; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #37474f; font-size: 1.1em;">Dry Run Support</strong>
            <p style="margin: 8px 0 0 0; color: #333;"><code>ALTER TABLE {foreign_table} SET {EXTERNAL|MANAGED} DRY RUN</code> - validates the migration without executing it.</p>
        </div>
    </div>
</div>

### Validation Checks

Run the following checks during the dry-run phase to catch issues before committing.

- Table format compatibility (Delta, Iceberg, etc.)
- Storage location permissions
- Schema compatibility
- Performance benchmarks

### Rollback Options

If a migration needs to be reversed, both the table state change and the data state can be rolled back.

- `UNSET MANAGED` command available for managed tables
- `RESTORE` for reverting to previous versions

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Common Considerations</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Managed tables require a UC storage location. Initial data copy may cause temporary latency. DBFS-backed tables require special handling. Use Python utilities for batch conversion of multiple tables.</p>
        </div>
    </div>
</div>

## Key Takeaways

- **Federation** provides a zero-downtime bridge for migrating to Unity Catalog
- **In-place migration** preserves namespaces, permissions, and query compatibility
- **Dry run** support enables safe validation before committing to migration
- Managed tables unlock the **full UC feature set** including Predictive Optimization

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>